<a href="https://colab.research.google.com/github/Dharshini13002/Deep-learning-codes-of-FER/blob/main/FER2013%2BGRUmodel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import torch.backends.cudnn as cudnn
import numpy as np
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
import time
import os
from PIL import Image
from tempfile import TemporaryDirectory

In [4]:
# Kaggle API setup
os.environ["KAGGLE_USERNAME"] = "dharshini1333333"  # Replace with your Kaggle username
os.environ["KAGGLE_KEY"] = "cecd150b72b85d04cc229a681a646b88"  # Replace with your Kaggle API key

In [5]:
from kaggle.api.kaggle_api_extended import KaggleApi

In [6]:
# Enable CuDNN benchmarking for performance
cudnn.benchmark = True
plt.ion()  # Enable interactive mode for matplotlib

In [7]:
# Use the existing Fer2013 dataset if present
data_dir = '/content/fer2013.csv'
if os.path.exists(data_dir):
    print("FER2013 dataset already exists at:", data_dir)
else:
    print("FER2013 dataset not found at:", data_dir)
    print("Downloading the dataset using Kaggle API...")
    api = KaggleApi()
    api.authenticate()
    api.dataset_download_files('deadskull7/fer2013', path='/content/', unzip=True)
    print("Dataset downloaded and unzipped successfully.")

FER2013 dataset not found at: /content/fer2013.csv
Dataset URL: https://www.kaggle.com/datasets/deadskull7/fer2013
Dataset downloaded and unzipped successfully.


In [28]:
# Data transformations for training and validation
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.5],  # Grayscale mean
                             [0.5])  # Grayscale std
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.5],  # Grayscale mean
                             [0.5])  # Grayscale std
    ]),
}

In [14]:
# Dataset Preparation
# ======================
class FER2013Dataset(Dataset):
    def __init__(self, csv_file, transform=None, usage="Training"):
        self.data = pd.read_csv(csv_file)
        self.data = self.data[self.data["Usage"] == usage]
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        pixels = self.data.iloc[idx]["pixels"]
        label = int(self.data.iloc[idx]["emotion"])
        pixels = np.array(pixels.split(), dtype=np.uint8).reshape(48, 48)

        img = Image.fromarray(pixels)

        if self.transform:
            img = self.transform(img)

        return img, label


In [15]:
# Transformations
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = FER2013Dataset("/content/fer2013.csv", transform=transform, usage="Training")
val_dataset = FER2013Dataset("/content/fer2013.csv", transform=transform, usage="PublicTest")
test_dataset = FER2013Dataset("/content/fer2013.csv", transform=transform, usage="PrivateTest")

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


In [16]:

# ======================
# GRU Model
# ======================
class GRUEmotion(nn.Module):
    def __init__(self, input_size=48, hidden_size=128, num_layers=2, num_classes=7):
        super(GRUEmotion, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # GRU processes sequences (rows of image)
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True, bidirectional=True)
        self.fc1 = nn.Linear(hidden_size * 2, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        # x: (batch, 1, 48, 48)
        x = x.squeeze(1)  # (batch, 48, 48)

        # treat each row (48 pixels) as one timestep
        out, _ = self.gru(x)  # (batch, seq_len=48, hidden*2)
        out = out[:, -1, :]   # last timestep output

        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)
        return out


In [17]:

# ======================
# Training Setup
# ======================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GRUEmotion().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [18]:
# ======================
# Training Loop
# ======================
def train_model(model, train_loader, val_loader, epochs=10):
    for epoch in range(epochs):
        model.train()
        train_loss, correct, total = 0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        train_acc = 100. * correct / total

        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()

        val_acc = 100. * val_correct / val_total

        print(f"Epoch [{epoch+1}/{epochs}], "
              f"Train Loss: {train_loss/len(train_loader):.4f}, Train Acc: {train_acc:.2f}%, "
              f"Val Loss: {val_loss/len(val_loader):.4f}, Val Acc: {val_acc:.2f}%")

train_model(model, train_loader, val_loader, epochs=20)


Epoch [1/20], Train Loss: 1.7208, Train Acc: 30.93%, Val Loss: 1.6664, Val Acc: 33.99%
Epoch [2/20], Train Loss: 1.6166, Train Acc: 36.62%, Val Loss: 1.5560, Val Acc: 39.06%
Epoch [3/20], Train Loss: 1.5269, Train Acc: 40.27%, Val Loss: 1.4958, Val Acc: 41.26%
Epoch [4/20], Train Loss: 1.4498, Train Acc: 43.50%, Val Loss: 1.4557, Val Acc: 42.91%
Epoch [5/20], Train Loss: 1.3737, Train Acc: 46.84%, Val Loss: 1.4220, Val Acc: 45.53%
Epoch [6/20], Train Loss: 1.2961, Train Acc: 50.07%, Val Loss: 1.4178, Val Acc: 45.17%
Epoch [7/20], Train Loss: 1.2126, Train Acc: 53.72%, Val Loss: 1.4230, Val Acc: 46.61%
Epoch [8/20], Train Loss: 1.1214, Train Acc: 57.41%, Val Loss: 1.4152, Val Acc: 46.92%
Epoch [9/20], Train Loss: 1.0283, Train Acc: 61.25%, Val Loss: 1.4143, Val Acc: 49.57%
Epoch [10/20], Train Loss: 0.9204, Train Acc: 65.38%, Val Loss: 1.5076, Val Acc: 48.45%
Epoch [11/20], Train Loss: 0.8171, Train Acc: 69.72%, Val Loss: 1.5667, Val Acc: 48.57%
Epoch [12/20], Train Loss: 0.7123, Train 

In [19]:
# ======================
# Testing
# ======================
def test_model(model, test_loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    print(f"Test Accuracy: {100.*correct/total:.2f}%")

test_model(model, test_loader)

Test Accuracy: 49.87%


In [1]:
import torch

def check_accuracy(model, loader, device):
    model.eval()   # evaluation mode
    correct, total = 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    acc = 100 * correct / total
    return acc

# =============================
# Example usage for train, val, test
# =============================
train_acc = check_accuracy(model, dataloaders['train'], device)
val_acc   = check_accuracy(model, dataloaders['val'], device)
test_acc  = check_accuracy(model, dataloaders['test'], device)

print(f"Train Accuracy: {train_acc:.2f}%")
print(f"Val Accuracy  : {val_acc:.2f}%")
print(f"Test Accuracy : {test_acc:.2f}%")


NameError: name 'model' is not defined

**Reasoning**:
Create PyTorch DataLoader instances for the training and testing datasets.

## Create dataloaders

### Subtask:
Create PyTorch `DataLoader` instances using the custom datasets for efficient batching during training and evaluation.

**Reasoning**:
Filter the dataframe into training and testing sets and create instances of the custom dataset with the appropriate transformations.

## Instantiate the custom dataset

### Subtask:
Create instances of the custom dataset for training and testing using the appropriate data splits from the CSV.

**Reasoning**:
Define a custom PyTorch Dataset class to handle the loading and transformation of the pixel data from the DataFrame.

## Create a custom dataset

### Subtask:
Define a custom PyTorch `Dataset` class that reads the image pixel data and labels from the DataFrame. This dataset will handle the transformation of pixel strings into image tensors.

**Reasoning**:
Load the `fer2013.csv` file into a pandas DataFrame and inspect its structure as requested by the subtask instructions.

## Load and inspect the csv data

### Subtask:
Load the `fer2013.csv` file into a pandas DataFrame and inspect its structure to understand how images and labels are stored.

# Task
Explain the error in the provided Python code for loading and processing the "fer2013.csv" dataset for a deep learning model. If possible, fix the error and provide the corrected code. Otherwise, diagnose the error and explain the potential causes. The code should load the dataset from "fer2013.csv", create PyTorch Datasets and DataLoaders for training and testing, and handle the image pixel data and labels.

## Load and inspect the csv data

### Subtask:
Load the `fer2013.csv` file into a pandas DataFrame and inspect its structure to understand how images and labels are stored.


**Reasoning**:
Load the `fer2013.csv` file into a pandas DataFrame and inspect its structure as requested by the subtask instructions.



In [10]:
import pandas as pd

# Load the dataset
df = pd.read_csv('/content/fer2013.csv')

# Display the first few rows
display(df.head())

# Print column names and data types
print(df.info())

# Print the shape of the DataFrame
print(df.shape)

,emotion,pixels,Usage
0,0,70 80 82 72 58 58 60 63 54 58 60 48 89 115 121...,Training
1,0,151 150 147 155 148 133 111 140 170 174 182 15...,Training
2,2,231 212 156 164 174 138 161 173 182 200 106 38...,Training
3,4,24 32 36 30 32 23 19 20 30 41 21 22 32 34 21 1...,Training
4,6,4 0 0 0 0 0 0 0 0 0 0 0 3 15 23 28 48 50 58 84...,Training


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35887 entries, 0 to 35886
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   emotion  35887 non-null  int64 
 1   pixels   35887 non-null  object
 2   Usage    35887 non-null  object
dtypes: int64(1), object(2)
memory usage: 841.2+ KB
None
(35887, 3)


## Create a custom dataset

### Subtask:
Define a custom PyTorch `Dataset` class that reads the image pixel data and labels from the DataFrame. This dataset will handle the transformation of pixel strings into image tensors.


**Reasoning**:
Define a custom PyTorch Dataset class to handle the loading and transformation of the pixel data from the DataFrame.



In [11]:
import numpy as np
from PIL import Image
from torch.utils.data import Dataset

class Fer2013Dataset(Dataset):
    def __init__(self, dataframe, transform=None):
        """
        Args:
            dataframe (pd.DataFrame): DataFrame containing the dataset.
            transform (callable, optional): Optional transform to be applied
                on a sample.
        """
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        row = self.dataframe.iloc[idx]
        pixels = row['pixels'].split(' ')
        pixels = np.array(pixels, dtype=np.uint8).reshape(48, 48)
        image = Image.fromarray(pixels, 'L') # 'L' mode for grayscale
        label = row['emotion']

        if self.transform:
            image = self.transform(image)

        return image, label

## Instantiate the custom dataset

### Subtask:
Create instances of the custom dataset for training and testing using the appropriate data splits from the CSV.


**Reasoning**:
Filter the dataframe into training and testing sets and create instances of the custom dataset with the appropriate transformations.



In [12]:
# Filter the DataFrame for training and testing data
df_train = df[df['Usage'] == 'Training']
df_test = df[df['Usage'] == 'PublicTest']

# Instantiate the custom dataset for training and testing
train_dataset = Fer2013Dataset(dataframe=df_train, transform=data_transforms['train'])
test_dataset = Fer2013Dataset(dataframe=df_test, transform=data_transforms['val'])

print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of testing samples: {len(test_dataset)}")

Number of training samples: 28709
Number of testing samples: 3589


## Create dataloaders

### Subtask:
Create PyTorch `DataLoader` instances using the custom datasets for efficient batching during training and evaluation.


**Reasoning**:
Create PyTorch DataLoader instances for the training and testing datasets.



In [13]:
from torch.utils.data import DataLoader

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print("Train DataLoader created.")
print("Test DataLoader created.")

Train DataLoader created.
Test DataLoader created.


## Summary:

### Data Analysis Key Findings

*   The `fer2013.csv` dataset was successfully loaded into a pandas DataFrame with 35887 rows and 3 columns (`emotion`, `pixels`, `Usage`).
*   The `pixels` column contains space-separated string representations of image pixel values.
*   A custom PyTorch `Fer2013Dataset` class was created to handle the conversion of the pixel strings into 48x48 grayscale images and provide the corresponding emotion labels.
*   The dataset was split into training (28709 samples) and testing (3589 samples) sets based on the 'Usage' column.
*   PyTorch `DataLoader` instances were successfully created for both the training and testing datasets with a batch size of 64.

### Insights or Next Steps

*   The next step is to define and train a deep learning model using the created `train_loader` and evaluate its performance using the `test_loader`.
*   Consider implementing data augmentation techniques within the `transform` parameter of the `Fer2013Dataset` to improve model generalization.
